In [0]:
import json
from pyspark.sql import functions as F

# 1. Parameterize paths (override via Databricks Job parameters)
dbutils.widgets.text("catalog_name", "workspace")
dbutils.widgets.text("landing_path", "/Volumes/workspace/bronze/raw_landing/qualtrics/responses/")
dbutils.widgets.text("checkpoint_path", "/Volumes/workspace/bronze/checkpoints/qualtrics_responses/")
dbutils.widgets.text("metadata_path", "/Volumes/workspace/bronze/raw_landing/qualtrics/metadata/dataset_metadata.json")

catalog = dbutils.widgets.get("catalog_name")
landing_path = dbutils.widgets.get("landing_path")
checkpoint_path = dbutils.widgets.get("checkpoint_path")
metadata_path = dbutils.widgets.get("metadata_path")
bronze_table = f"{catalog}.bronze.qualtrics_responses_raw"

# 2. Validate source contract from dataset_metadata.json before ingesting
with open(metadata_path, "r") as f:
    meta = json.load(f)

expected_cols = set(meta["columns"])
print(f"Contract loaded for '{meta['dataset_name']}' | Expected columns: {len(expected_cols)}")

# 3. Incremental ingestion using Auto Loader (cloudFiles)
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("rescuedDataColumn", "_rescued_data")
    .load(landing_path)
)

# 4. Verify contract columns exist
missing_cols = expected_cols - set(raw_stream.columns)
assert not missing_cols, f"Source contract violation! Missing columns: {missing_cols}"

# 5. Append production lineage & audit metadata
bronze_df = (
    raw_stream
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# 6. Write append-only Bronze Delta table
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_path}/data")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)
query.awaitTermination()

print(f"Bronze row count: {spark.table(bronze_table).count()}")

Contract loaded for 'synthetic_qualtrics_responses' | Expected columns: 17
Bronze row count: 120
